# Prosody Energy Feature Extraction (IEMOCAP)

This notebook extracts RMS energy features with silence-aware masking.
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_extraction.common import machine_name_from_env, resolve_thread_workers

MACHINE_NAME = machine_name_from_env()
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "prosody_energy"
OUT_FILE = "prosody_energy_features.csv"

# Audio + feature params
TARGET_SR = 16_000
FRAME_LENGTH = 2048
HOP_LENGTH = 256
RMS_DB_THRESHOLD = -50.0
RMS_DB_PERCENTILE = 10.0
SILENCE_TOP_DB = 40.0
MIN_VOICED_FRAMES = 3

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/prosody_energy/prosody_energy_features.csv')

In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def _drop_short_runs(mask: np.ndarray, min_run: int) -> np.ndarray:
    # Drop short True runs shorter than min_run frames
    if min_run <= 1:
        return mask

    cleaned = mask.copy()
    run_start = None
    for idx, is_true in enumerate(mask):
        if is_true and run_start is None:
            run_start = idx
        if not is_true and run_start is not None:
            run_len = idx - run_start
            if run_len < min_run:
                cleaned[run_start:idx] = False
            run_start = None
    if run_start is not None:
        run_len = len(mask) - run_start
        if run_len < min_run:
            cleaned[run_start:] = False
    return cleaned


def compute_energy(audio: np.ndarray, sr: int) -> tuple[np.ndarray, np.ndarray]:
    # RMS energy with silence-aware masking
    rms = librosa.feature.rms(
        y=audio,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )[0]
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    if RMS_DB_PERCENTILE is None:
        rms_db_effective = RMS_DB_THRESHOLD
    else:
        rms_db_effective = max(
            RMS_DB_THRESHOLD,
            float(np.percentile(rms_db, RMS_DB_PERCENTILE)),
        )

    energy_mask = rms_db >= rms_db_effective
    nonsilent_intervals = librosa.effects.split(
        audio,
        top_db=SILENCE_TOP_DB,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )
    nonsilent_mask = np.zeros_like(energy_mask, dtype=bool)
    for start, end in nonsilent_intervals:
        start_frame = start // HOP_LENGTH
        end_frame = int(np.ceil(end / HOP_LENGTH))
        nonsilent_mask[start_frame:end_frame] = True
    energy_mask &= nonsilent_mask
    energy_mask = _drop_short_runs(energy_mask, MIN_VOICED_FRAMES)

    rms_db_masked = rms_db.copy()
    rms_db_masked[~energy_mask] = np.nan
    return rms_db_masked, energy_mask


def summarize_curve(prefix: str, values: np.ndarray) -> dict[str, float]:
    # Summary stats ignoring NaNs
    vec = np.asarray(values, dtype=float).ravel()
    vec = vec[~np.isnan(vec)]
    if vec.size == 0:
        return {
            f"{prefix}_mean": float("nan"),
            f"{prefix}_std": float("nan"),
            f"{prefix}_min": float("nan"),
            f"{prefix}_max": float("nan"),
            f"{prefix}_median": float("nan"),
        }
    return {
        f"{prefix}_mean": float(vec.mean()),
        f"{prefix}_std": float(vec.std()),
        f"{prefix}_min": float(vec.min()),
        f"{prefix}_max": float(vec.max()),
        f"{prefix}_median": float(np.median(vec)),
    }


def extract_energy_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    rms_db_masked, mask = compute_energy(audio, sr)
    voiced_ratio = float(np.sum(mask)) / float(max(1, mask.size))
    features = {
        "energy_frames": float(rms_db_masked.size),
        "energy_voiced_ratio": float(voiced_ratio),
    }
    features.update(summarize_curve("energy_rms_db", rms_db_masked))
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(9887, 7)

In [5]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = resolve_thread_workers(MACHINE_NAME)
PROGRESS_MIN_INTERVAL = 1.0

print(
    f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | "
    f"machine={MACHINE_NAME} | workers={NUM_WORKERS}"
)


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_energy_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Compute device: cpu | extractor_backend=cpu | machine=macbook | workers=6


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

Saved: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/prosody_energy/prosody_energy_features.csv
Workers used: 6 (cpu_count=8)


(9887, 15)